# 📋 Projeto Final — Sistemas Baseados em Conhecimento
## Sistema Inteligente de Monitoramento de Impressão 3D

**Disciplina:** Sistemas Baseados em Conhecimento  
**Professor:** Carlos Roque / Paulo Brandão / Caíque Zanetti
**Prazo de entrega:** Aula 10 — Apresentação  
**Formato:** Grupos (definidos na Aula 6)

---

## Contexto

Na Aula 6 vocês construíram o mecanismo completo de integração entre modelos de IA e um Motor de Inferência. O pipeline funcionou com dados reais — chamada à API do Azure Custom Vision, classificador SVM de vibração e contexto operacional via CSV.

Mas o sistema ainda era uma **amostra**: 4 peças, 9 regras, cobertura parcial das classes, sem decisão gradual.

O projeto final é transformar essa amostra em um **sistema real e completo**.

---

## O que vocês vão construir

Um Sistema Baseado em Conhecimento para monitoramento de impressão 3D que:

- Consome dados reais dos modelos treinados na aula de RP (incluindo Classificação e Detecção de Anomalias/Drift da Aula 8)
- Cobre **todas as classes** detectadas pelo Azure Custom Vision
- Implementa **decisão gradual** com score de risco ponderado
- Analisa pelo menos **10 peças reais** com e sem defeito
- Produz decisões **explicáveis** com `por_que()` documentado
- É apresentado e defendido pelo grupo na Aula 10

---

## Entregáveis

### 1. Notebook (`projeto_final_grupoN.ipynb`)

O notebook deve conter o pipeline completo e funcional:

**Pipeline de dados**
- Upload e chamada real à API do Azure Custom Vision
- Upload e uso do SVM de vibração com o scaler
- Upload e integração do modelo de detecção de anomalias por PCA (Aula 8 RP)
- Contexto operacional em CSV com pelo menos 10 peças (com e sem defeito, variando criticidade e prazo)

**Grounding**
- Funções de grounding para visão, vibração e contexto
- Limiares **definidos e justificados** pelo grupo — não apenas copiados da Aula 6
- Explicação de por que escolheram aquele valor de corte

**Score de Risco**
- Função `calcular_score()` com pesos definidos pelo grupo
- Tabela de pesos com justificativa para cada sinal
- Faixas de decisão definidas e justificadas

**Base de Conhecimento**
- Cobertura completa das 5 classes do Azure:
  - `leg_broken`
  - `no_support`
  - `bed_no_stick`
  - `no_bottom`
  - `no_defected`
- As 5 ações graduais presentes: GO / MONITORAR / AJUSTAR / INVESTIGAR / NOGO
- Inclusão de regra(s) que lidem com a detecção de anomalia via PCA (drift na vibração)
- Cada regra com `justificativa` preenchida — não deixar em branco

**Motor e Resultados**
- Motor rodando para todas as peças
- Tabela comparativa final
- `por_que()` documentado

**Reflexão do grupo**
- Quais foram as decisões de engenharia mais difíceis?
- Em algum caso o motor gerou `decisao(INDEFINIDA)`? Por quê?
- O que mudariam se tivessem mais dados ou mais tempo?

*Apresentação (até 25 minutos por grupo)*
---
---

## Classes do Azure que precisam de cobertura

A Aula 6 cobriu apenas `leg_broken`. As demais ficam como lacuna que vocês devem preencher:

| Classe | O que significa | Perguntas para pensar |
|---|---|---|
| `leg_broken` | Fratura estrutural na peça | Já coberta na Aula 6 — revisar e expandir |
| `no_support` | Suporte ausente ou mal formado | Para ou ajusta? Depende da criticidade? |
| `bed_no_stick` | Problema de adesão à base | É grave o suficiente para NOGO? |
| `no_bottom` | Fundo da peça ausente | O que muda se a vibração também indicar erro? |
| `no_defected` | Sem defeito detectado | E se a vibração discordar? Quem você confia? |

---

## Score de Risco — ponto de partida

A Aula 5 introduziu o conceito de score ponderado. Usem como referência e **adaptem** para as classes reais do Azure:

| Sinal | Peso sugerido | Vocês podem mudar? |
|---|---|---|
| `leg_broken` detectado | 45 | ✅ Sim — justifique |
| `no_support` detectado | ? | Vocês definem |
| `bed_no_stick` detectado | ? | Vocês definem |
| `no_bottom` detectado | ? | Vocês definem |
| `confianca(alta)` — bônus | 15 | ✅ Sim — justifique |
| `vibracao(erro)` | 40 | ✅ Sim — justifique |
| `anomalia_pca(detectada)` | 30 | ✅ Sim — justifique |
| `criticidade(alta)` | 20 | ✅ Sim — justifique |
| `prazo(curto)` | 10 | ✅ Sim — justifique |

**Faixas de decisão sugeridas** (podem ser alteradas com justificativa):

| Score | Ação |
|---|---|
| 0 – 29 | GO |
| 30 – 49 | MONITORAR |
| 50 – 69 | AJUSTAR |
| 70 – 84 | INVESTIGAR |
| 85+ | NOGO |

---

## Critérios de Avaliação

| Critério | Peso | O que será observado |
|---|---|---|
| **Pipeline completo e funcional** | 25% | Todas as etapas rodando sem erros |
| **Cobertura das regras** | 20% | Todas as classes com resposta, sem INDEFINIDA evitável |
| **Decisão gradual** | 20% | Score implementado, 5 ações presentes e proporcionais |
| **Explicabilidade** | 20% | `por_que()` documentado, justificativas preenchidas |
| **Defesa das escolhas** | 15% | Limiares, pesos e prioridades justificados — não chutados |

---

## Dicas importantes

**Sobre os dados**
- Usem imagens variadas — peças com defeito e sem defeito, diferentes classes
- O `contexto_pecas.csv` deve refletir situações reais: misture criticidade alta e média, prazo curto e longo
- A leitura de vibração ainda é sorteada do CSV — isso é aceitável, mas expliquem na apresentação o que mudaria com o sensor real
- O pipeline deve utilizar o PCA treinado para verificar o erro de reconstrução e acionar regras de anomalia, caso ocorra drift.

**Sobre as regras**
- Uma regra sem justificativa é uma regra que não pode ser auditada — preencha sempre
- Conflito de sensores (visão diz uma coisa, vibração diz outra) é o caso mais rico — não ignorem
- Se o motor gerar `INDEFINIDA` para alguma peça, isso é um bug de cobertura — resolvam ou documentem

**Sobre o score**
- Os pesos são uma decisão de engenharia — numa fábrica real viriam de análise de falha (FMEA)
- Não há resposta certa, mas há respostas sem justificativa — evitem
- Grupos diferentes vão ter pesos diferentes — isso é esperado e vai tornar a apresentação interessante

---

## Arquivos que vocês já têm (Aulas 6 e 8)

- `vibration_features.csv` — dataset de vibração do professor Paulo
- `svm_model.joblib` + `scaler.joblib` — modelos treinados
- `pca_model.joblib`, `pca_reference.joblib`, `reducao_config.joblib` — modelos de redução e anomalias (Aula 8)
- Notebook da Aula 6 — pipeline base para partir

**Novos arquivos que vocês vão criar:**
- `contexto_pecas.csv` expandido — pelo menos 10 peças
- `projeto_final_grupoN.ipynb` — notebook do projeto
- Apresentação em slides

---

*Qualquer dúvida técnica durante o desenvolvimento — tragam para a aula. Bom trabalho!*
